# SAC — Soft Actor-Critic : controle continu off-policy avec maximisation d'entropie

**Algorithme** : SAC (Soft Actor-Critic)  
**Environnement** : HalfCheetah-v5 (MuJoCo, Gymnasium)  
**Papiers** : [Haarnoja et al., 2018](https://arxiv.org/abs/1801.01290) (SAC) ; [Haarnoja et al., 2018](https://arxiv.org/abs/1812.05905) (SAC avec alpha auto-tune)

> **Quick Facts** (inspires de [SpinningUp](https://spinningup.openai.com)) :
> - SAC est un algorithme **off-policy** pour les espaces d'action **continus**.
> - C'est une approche a **maximum d'entropie** : l'agent maximise `r + alpha * H(pi)`.
> - L'exploration est **intrinseque** (pas de bruit externe comme DDPG/TD3).
> - La temperature `alpha` peut etre **apprise automatiquement**.
> - SAC utilise des **twin critics** (comme TD3) mais **pas de target actor**.

| Propriete | DDPG | TD3 | SAC |
|-----------|------|-----|-----|
| Model-free | oui | oui | oui |
| Off-policy | oui | oui | oui |
| Politique deterministe | oui | oui | **non** |
| Replay buffer | oui | oui | oui |
| Twin critics | non | oui | oui |
| Target actor | oui | oui | **non** |
| Exploration | bruit externe | bruit externe | **entropie intrinseque** |
| Temperature alpha | non | non | **oui (apprise)** |

## Carte mentale : ce que SAC reprend et ce qu'il change

Avant de rentrer dans les equations, il faut situer SAC dans le parcours du projet.

| Idee deja vue | Ou on l'a vue | Ce que SAC reprend | Ce que SAC change |
|---|---|---|---|
| Replay buffer | DQN, DDPG, TD3 | Reutiliser les transitions plusieurs fois | Cibles Bellman deviennent *soft* avec entropie |
| Off-policy | Q-learning, DQN, DDPG, TD3 | Apprendre depuis d'anciennes politiques | L'action cible vient de la politique courante |
| Actor-critic | REINFORCE, A2C, PPO | Actor + critic | Critic est Q(s,a), pas V(s) |
| Twin critics | TD3 | min(Q1, Q2) contre l'overestimation | Le min est combine avec le bonus d'entropie |
| Politique stochastique | REINFORCE, A2C, PPO | Optimiser une distribution d'actions | Distribution continue, squashee par tanh |

**L'intuition courte** : DQN nous a donne le replay buffer, TD3 les twin critics, PPO/A2C une politique stochastique. SAC assemble ces briques et ajoute une idee centrale : maximiser aussi l'entropie.

DDPG/TD3 explorent en ajoutant du bruit a une politique deterministe. SAC apprend directement une politique probabiliste : l'exploration n'est plus un bricolage externe, elle fait partie de l'objectif optimise.

In [ ]:
import copy
import json
import math
from collections import deque
from pathlib import Path

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

try:
    from IPython.display import Video, display
except Exception:  # pragma: no cover - optionnel
    Video = display = None

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
env = gym.make("HalfCheetah-v5")
OBS_DIM = env.observation_space.shape[0]
ACTION_DIM = env.action_space.shape[0]
ACTION_LOW = env.action_space.low.astype(np.float32)
ACTION_HIGH = env.action_space.high.astype(np.float32)

print("Observation space :", env.observation_space)
print("Action space      :", env.action_space)
print("OBS_DIM           :", OBS_DIM)
print("ACTION_DIM        :", ACTION_DIM)

env.close()

## Transition : l'innovation de SAC tient en un mot — entropie

DDPG et TD3 explorent en ajoutant du bruit **exterieur** a une politique deterministe. Ce bruit est aveugle : il ne sait pas quels etats sont incertains, et son echelle doit etre tunee manuellement.

SAC change l'objectif d'apprentissage. Plutot que de maximiser seulement la recompense, l'agent maximise la **recompense augmentee par l'entropie** de sa propre politique. L'exploration emerge alors **naturellement** de l'objectif, sans aucun bruit externe.

La section suivante construit l'intuition de l'entropie pas a pas avant d'ecrire la moindre equation.

## 1. Qu'est-ce que l'entropie ?

### Definition

Pour une distribution discrete $\pi$ sur un ensemble d'actions, l'entropie est :

$$\mathcal{H}(\pi) = \mathbb{E}_{a \sim \pi}[-\log \pi(a)] = -\sum_a \pi(a) \log \pi(a)$$

**Intuition.** Pensez a une piece de monnaie biaisee a 99% : on peut presque toujours predire le resultat — faible entropie. Une piece equilibree a 50/50 est impredictible — entropie maximale.

- Distribution **deterministe** (toute la masse sur une action) : $\mathcal{H} = 0$
- Distribution **uniforme** sur $n$ actions : $\mathcal{H} = \log n$ (maximum possible)

Plus une politique est etalee et impredictible, plus son entropie est elevee.

In [ ]:
# Entropie de 4 politiques discretes (3 actions : A, B, C)

def entropy(probs):
    probs = np.clip(probs, 1e-10, 1.0)
    return -np.sum(probs * np.log(probs))

policies = {
    'Tres deterministe': np.array([0.001, 0.998, 0.001]),
    'Deterministe':      np.array([0.01,  0.98,  0.01]),
    'Equilibree':        np.array([0.20,  0.60,  0.20]),
    'Uniforme':          np.array([0.33,  0.34,  0.33]),
}
max_H = np.log(3)

print(f"{'Politique':<22} {'pi(A)':<7} {'pi(B)':<7} {'pi(C)':<7} {'H(pi)':<8} {'% max'}")
print("-" * 62)
for name, p in policies.items():
    h = entropy(p)
    print(f"{name:<22} {p[0]:<7.3f} {p[1]:<7.3f} {p[2]:<7.3f} {h:<8.4f} {h/max_H*100:.1f}%")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#4ECDC4', '#45B7D1', '#FF6B6B']
for ax, (name, p) in zip(axes, policies.items()):
    h = entropy(p)
    ax.bar(['A', 'B', 'C'], p, color=colors, alpha=0.85, edgecolor='k', lw=1.1)
    ax.axhline(1/3, color='gray', ls='--', lw=1, alpha=0.6)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{name}\nH={h:.3f} ({h/max_H*100:.0f}% max)", fontsize=10, fontweight='bold')
    ax.set_ylabel('pi(a|s)')
plt.suptitle('Quatre politiques discretes et leur entropie', fontweight='bold')
plt.tight_layout()
plt.show()

### Lecture : de la politique déterministe à la politique uniforme

- **Tres deterministe** : 99.8% sur B, entropie quasi nulle. L'agent n'explore pas.
- **Deterministe** : 98% sur B, entropie 10% du max. Presque aucune exploration.
- **Equilibree** : 60%/20%/20%, entropie 87% du max. Bon compromis.
- **Uniforme** : ~33% sur chaque action, entropie maximale. Exploration totale.

**Retenir** : l'entropie mesure la diversite d'une politique. Une politique deterministe a $H=0$, une politique uniforme atteint $H=\log(n)$. SAC recompense le milieu : etre explore sans etre chaotique.

## 2. L'objectif entropy-regularized

SAC remplace l'objectif standard $J(\pi) = \mathbb{E}[\sum \gamma^t r_t]$ par :

$$\boxed{J(\pi) = \mathbb{E}\left[\sum_{t=0}^\infty \gamma^t \left(r_t + \alpha \cdot \mathcal{H}(\pi(\cdot|s_t))\right)\right]}$$

Le terme $\alpha \cdot \mathcal{H}(\pi(\cdot|s_t))$ est un **bonus d'entropie** ajoute a chaque pas.
- $\alpha > 0$ est la **temperature** : elle controle l'importance de ce bonus.
- **$\alpha$ eleve** : l'agent prefere les politiques etalees (exploration).
- **$\alpha$ faible** : la recompense domine, l'agent devient quasi-deterministe (exploitation).

**Consequence directe.** Dans un etat ou plusieurs actions sont egalement bonnes, la politique optimale SAC repartit uniformement la masse — ce qui maximise l'entropie sans perdre de recompense. L'exploration emerge de l'objectif, sans bruit externe.

In [ ]:
# Effet de alpha sur l'objectif total : E[r] + alpha * H(pi)

policy_names = ['Deterministe (R=10, H=0.11)', 'Equilibree (R=8.5, H=0.95)', 'Uniforme (R=5.0, H=1.10)']
rewards   = [10.0, 8.5, 5.0]
entropies = [0.112, 0.951, 1.099]

alpha_vals = np.linspace(0, 5, 200)
colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(10, 5))
for name, R, H, c in zip(policy_names, rewards, entropies, colors):
    ax.plot(alpha_vals, [R + a*H for a in alpha_vals], lw=2.5, color=c, label=name)

ax.set_xlabel('Temperature alpha', fontsize=12, fontweight='bold')
ax.set_ylabel('Objectif total J = E[R] + alpha*H(pi)', fontsize=11)
ax.set_title("Impact de alpha sur l'objectif SAC", fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'Politique':<30} {'alpha=0':<12} {'alpha=1':<12} {'alpha=3'}")
print("-" * 66)
for name, R, H in zip(policy_names, rewards, entropies):
    print(f"{name:<30} {R:<12.2f} {R+H:<12.3f} {R+3*H:.3f}")

### Lecture : quand l'entropie change le classement

- **alpha = 0** : seule la recompense compte. La politique deterministe (R=10) domine.
- **alpha petit (~0.5)** : la politique equilibree commence a devenir competitive.
- **alpha grand (>2)** : l'entropie domine. La politique uniforme peut l'emporter.

Le point d'intersection des courbes montre quand investir dans l'exploration rapporte plus que d'optimiser la recompense instantanee. En pratique, ce point change au fil de l'apprentissage — c'est pourquoi SAC apprend alpha automatiquement.

## Transition : mais comment choisir alpha ?

Fixer alpha manuellement est fragile : la valeur optimale change au cours de l'apprentissage. Au debut, l'agent doit explorer (alpha grand). A la fin, il doit exploiter (alpha petit).

SAC resout cela en **apprenant alpha automatiquement** via une contrainte sur l'entropie minimale.

## 3. Auto-tuning de alpha

### La contrainte d'entropie

Au lieu de fixer alpha, on impose que l'entropie moyenne de la politique ne descende pas sous un seuil $\bar{\mathcal{H}}$ :

$$\mathcal{L}(\alpha) = \mathbb{E}_{a \sim \pi}\left[-\alpha \left(\log \pi(a|s) + \bar{\mathcal{H}}\right)\right]$$

**Interpretation.** Si l'entropie courante est **superieure** a $\bar{\mathcal{H}}$, la loss pousse alpha a **diminuer** (moins d'exploration). Si l'entropie est **inferieure**, la loss pousse alpha a **augmenter** (plus d'exploration). Alpha fonctionne comme un thermostat automatique.

**La cible standard.** L'heuristique recommandee par SpinningUp est $\bar{\mathcal{H}} = -\dim(\mathcal{A})$. Pour HalfCheetah ($\dim(\mathcal{A}) = 6$), `target_entropy = -6`. En pratique, on parametrise $\alpha = \exp(\log\alpha)$ et on optimise $\log\alpha$ avec Adam, ce qui garantit $\alpha > 0$.

In [ ]:
# Simulation de l'auto-tuning d'alpha sur un entrainement jouet

np.random.seed(42)
n_steps = 800
steps = np.arange(n_steps)
entropy_target = 0.8
alpha_lr = 0.003

entropy_sim = np.zeros(n_steps)
alpha_sim   = np.zeros(n_steps)
alpha_curr  = 1.5

for i in range(n_steps):
    natural_entropy = 1.2 * np.exp(-i / 400) + 0.3 + 0.04 * np.random.randn()
    entropy_sim[i]  = natural_entropy + 0.3 * (alpha_curr - 1.0)
    alpha_grad = -(np.log(max(entropy_sim[i], 1e-4)) + np.log(entropy_target))
    alpha_curr = np.clip(alpha_curr - alpha_lr * alpha_grad, 0.05, 4.0)
    alpha_sim[i] = alpha_curr

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(steps, entropy_sim, 'royalblue', lw=2, label='Entropie H(pi)')
axes[0].axhline(entropy_target, color='k', ls='--', lw=1.5, label=f'Cible = {entropy_target}')
axes[0].fill_between(steps, entropy_target-0.05, entropy_target+0.05, alpha=0.15, color='k')
axes[0].set_ylabel('Entropie H(pi)')
axes[0].set_title('Auto-tuning alpha : entropie converge vers la cible', fontweight='bold')
axes[0].legend()

axes[1].plot(steps, alpha_sim, 'darkorange', lw=2, label='alpha (temperature)')
axes[1].axhline(1.0, color='gray', ls=':', lw=1.2, label='alpha initial=1.5')
axes[1].set_xlabel("Pas d'entrainement")
axes[1].set_ylabel('alpha')
axes[1].set_title("Alpha s'ajuste pour maintenir l'entropie autour de la cible")
axes[1].legend()

plt.tight_layout()
plt.show()

### Lecture : alpha comme régulateur

- **Phase initiale** : l'entropie depasse la cible → alpha diminue (moins d'exploration).
- **Phase mediane** : l'entropie descend sous la cible → alpha augmente pour forcer l'exploration.
- **Convergence** : alpha se stabilise quand l'entropie oscille autour de la cible.

Ce comportement est celui d'un **regulateur** : alpha joue le role d'un multiplicateur de Lagrange qui equilibre recompense et entropie automatiquement.

### Recapitulatif : entropie et alpha dans SAC

| Concept | Formule | Intuition |
|---------|---------|----------|
| Entropie | $\mathcal{H}(\pi) = \mathbb{E}[-\log\pi(a\|s)]$ | Mesure d'incertitude de la politique |
| Objectif SAC | $J(\pi) = \mathbb{E}[R] + \alpha \cdot \mathcal{H}(\pi)$ | Recompense + bonus d'exploration |
| alpha faible | $\alpha \to 0$ | Exploitation (comme DDPG) |
| alpha eleve | $\alpha$ grand | Exploration |
| Auto-tune | $\mathcal{L}(\alpha) = -\alpha(\log\pi + \bar{\mathcal{H}})$ | alpha apprend a maintenir $H \geq \bar{\mathcal{H}}$ |
| Cible standard | $\bar{\mathcal{H}} = -\dim(\mathcal{A})$ | Heuristique empirique MuJoCo |

## Transition : comment estimer Q avec entropie ?

On sait que l'agent doit maximiser $r + \alpha H(\pi)$. Pour apprendre a le faire avec du Q-learning, il faut adapter l'equation de Bellman pour inclure l'entropie. C'est ce que fait la **soft Bellman equation**.

## 4. Soft Bellman equation et V(s) implicite

### De la cible standard a la cible soft

Dans DDPG/TD3, la cible Bellman est : $y = r + \gamma (1-d) \min Q_{targ}(s', \tilde{a}')$

Dans SAC, la cible inclut le terme d'entropie :

$$\boxed{y = r + \gamma(1-d)\left[\min_{i=1,2} Q_{targ}(s', \tilde{a}') - \alpha \log\pi(\tilde{a}'|s')\right]}$$

ou $\tilde{a}' \sim \pi_\theta(\cdot|s')$ est echantillonnee depuis la **politique courante** (pas une cible figee).

**Terme par terme.**
- $\min Q_{targ}(s', \tilde{a}')$ : valeur estimee de la prochaine action (anti-overestimation).
- $-\alpha \log\pi(\tilde{a}'|s')$ : bonus pour les actions a faible probabilite. Le log-prob est negatif, donc $-\alpha\log\pi > 0$ pour les actions improbables. La politique est poussee a garder une queue longue.

In [ ]:
# Comparaison : cible Bellman standard vs cible soft (scenario fictif)

np.random.seed(7)
n_actions = 5
action_labels = [f'a{i}' for i in range(n_actions)]

q_values  = np.array([4.0, 6.5, 3.0, 7.0, 2.5])
log_probs = np.array([-0.5, -1.8, -0.3, -2.5, -0.2])  # log pi(a|s'), toujours negatif
alpha = 0.5

standard_targets = q_values
soft_targets     = q_values - alpha * log_probs  # bonus pour actions improbables

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x, w = np.arange(n_actions), 0.35

axes[0].bar(x - w/2, standard_targets, w, label='Standard (TD3)', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, soft_targets, w, label='Soft (SAC)', color='tomato', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(action_labels)
axes[0].set_ylabel('Valeur de la cible y')
axes[0].set_title('Cibles Bellman : standard vs soft', fontweight='bold')
axes[0].legend()

diff = soft_targets - standard_targets
bar_colors = ['#e74c3c' if d < 0 else '#2ecc71' for d in diff]
axes[1].bar(x, diff, color=bar_colors, alpha=0.8)
axes[1].axhline(0, color='k', lw=1)
axes[1].set_xticks(x); axes[1].set_xticklabels(action_labels)
axes[1].set_ylabel('Difference (soft - standard)')
axes[1].set_title('Delta = -alpha * log_pi(a|s)', fontweight='bold')
for i, (d, lp) in enumerate(zip(diff, log_probs)):
    axes[1].text(i, d + 0.03*np.sign(d), f'logpi={lp:.1f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()
print('Les actions improbables (log_pi tres negatif) recoivent un bonus positif dans la cible soft.')
print('La politique est ainsi encouragee a explorer meme les actions rarement choisies.')

**Lecture.** Le terme $-\alpha\log\pi(a\mid s')$ est positif puisque $\log\pi$ est généralement négatif. Il relève donc davantage la cible des actions que la politique juge encore improbables. SAC ne remplace pas la valeur $Q$ par l'entropie : il optimise leur compromis. Quand $\alpha$ diminue, la cible se rapproche progressivement de la cible déterministe utilisée par TD3.

### V(s) implicite : pas de reseau separe

Le papier SAC original (2018a) entrainait un reseau separe pour $V(s)$. La version moderne (SAC v2, 2018b) montre qu'on peut l'eliminer :

$$V(s') = \mathbb{E}_{\tilde{a}' \sim \pi}[\min(Q_1(s',\tilde{a}'), Q_2(s',\tilde{a}')) - \alpha \log \pi(\tilde{a}'|s')]$$

Cette expression est calculee a la volee pendant le calcul de la cible Bellman, sans reseau supplementaire. On n'a besoin que de deux Q-networks + un actor.

**Transition.** Les equations sont posees. Il est temps de construire les composants reseau un a un, en testant chacun avant de passer au suivant.

## 5. Twin critics

### Pourquoi deux Q-networks independants ?

SAC emprunte les **twin critics** de TD3 pour la meme raison : reduire le biais d'overestimation. Un seul Q-network tend a surestimer les valeurs d'action, ce qui fait converger l'actor vers des actions surestimees plutot que les meilleures.

Avec deux Q-networks entraines independamment, on utilise le **minimum** pour les cibles Bellman :

$$\min(Q_1(s', \tilde{a}'), Q_2(s', \tilde{a}'))$$

### 3 differences structurelles avec TD3

| Aspect | TD3 | SAC |
|--------|-----|-----|
| **Action dans la cible** | $\mu_{targ}(s') + \epsilon$ clipped | $\tilde{a}' \sim \pi_\theta(\cdot|s')$ courante |
| **Cible Bellman** | $\min Q_{targ}(s', \tilde{a}')$ | $\min Q_{targ}(s', \tilde{a}') - \alpha\log\pi(\tilde{a}'|s')$ |
| **Loss acteur** | $-\mathbb{E}[Q_1(s, \mu_\theta(s))]$ | $-\mathbb{E}[\min Q(s, \tilde{a}) - \alpha\log\pi(\tilde{a}|s)]$ |

Ces trois differences font que SAC n'a plus besoin de target actor ni de policy delay.

In [ ]:
def _orthogonal_init(layer, gain=1.0):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.zeros_(layer.bias)
    return layer


class ContinuousQNetwork(nn.Module):
    """Q(s, a; phi) — MLP qui prend [s; a] en entree et produit un scalaire.

    Architecture : (obs_dim + action_dim) -> hidden(ReLU) -> hidden(ReLU) -> 1
    Identique au ContinuousQNetwork de DDPG/TD3.
    """

    def __init__(self, obs_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.fc1 = _orthogonal_init(nn.Linear(obs_dim + action_dim, hidden_dim), gain=np.sqrt(2))
        self.fc2 = _orthogonal_init(nn.Linear(hidden_dim, hidden_dim), gain=np.sqrt(2))
        self.q_out = _orthogonal_init(nn.Linear(hidden_dim, 1), gain=1.0)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.q_out(x).squeeze(-1)  # (batch,)


class TwinQNetwork(nn.Module):
    """Paire de Q-networks independants — anti-overestimation via min(Q1, Q2).

    Identique au TwinQNetwork de TD3, reutilise sans modification dans SAC.
    """

    def __init__(self, obs_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.q1 = ContinuousQNetwork(obs_dim, action_dim, hidden_dim)
        self.q2 = ContinuousQNetwork(obs_dim, action_dim, hidden_dim)

    def forward(self, state, action):
        return self.q1(state, action), self.q2(state, action)

In [ ]:
# Mini-test : TwinQNetwork — shapes et independance Q1 != Q2

torch.manual_seed(1)
_obs_dim, _act_dim = 17, 6
twin_q = TwinQNetwork(_obs_dim, _act_dim, hidden_dim=64)

_states  = torch.randn(8, _obs_dim)
_actions = torch.randn(8, _act_dim)
_q1, _q2 = twin_q(_states, _actions)

assert _q1.shape == (8,), f'Q1 shape: {_q1.shape}'
assert _q2.shape == (8,), f'Q2 shape: {_q2.shape}'
assert not torch.allclose(_q1, _q2), 'Q1 et Q2 doivent etre differents'

print(f'Q1 shape : {_q1.shape}  OK')
print(f'Q2 shape : {_q2.shape}  OK')
print(f'Q1 sample : {_q1[:3].detach().numpy().round(3)}')
print(f'Q2 sample : {_q2[:3].detach().numpy().round(3)}')
print(f'Q1 != Q2  : OK (max diff = {(_q1 - _q2).abs().max().item():.4f})')

## Transition : du critic a l'actor

Le critic sait evaluer des actions. L'actor doit **choisir** des actions — et pour SAC, il doit les choisir de facon stochastique avec un gradient bien defini. C'est le role du Squashed Gaussian Actor.

## 6. L'acteur Squashed Gaussian

### Architecture : trunk + deux tetes

L'actor SAC est un **Squashed Gaussian Actor** : un MLP partage qui produit une distribution gaussienne sur les actions, dont la variance depend de l'etat :

$$\text{MLP}(s) \to h \to \begin{cases} \mu_\theta(s) \in \mathbb{R}^{\dim(A)} \\ \log\sigma_\theta(s) \in \mathbb{R}^{\dim(A)} \end{cases}$$

**Pourquoi log_std dependant de l'etat ?** Un log_std global ne peut pas encoder l'incertitude par etat. Dans des etats bien compris, la variance doit etre faible. Dans des etats ambigus, elle doit etre elevee. SpinningUp note que la version avec log_std global ne fonctionne pas bien pour SAC.

### Le reparameterization trick

Pour que le gradient de la loss acteur passe a travers l'echantillonnage, SAC utilise :

$$u = \mu_\theta(s) + \sigma_\theta(s) \odot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

On echantillonne $\varepsilon$ de facon deterministe (pas de gradient a travers $\varepsilon$), et $u$ est une fonction differentiable de $\theta$. Le gradient peut donc remonter jusqu'a $\theta$.

### Tanh squashing et correction du log-prob

Les actions doivent etre bornees dans $[-1, 1]^{\dim(A)}$. On applique $\tanh$ a $u$ : $a = \tanh(u)$.

Mais $\tanh$ change la distribution. Le log-prob doit etre corrige par le Jacobien du changement de variable :

$$\boxed{\log\pi(a|s) = \log\pi_{\text{gauss}}(u|s) - \sum_{i=1}^{d} \log(1 - \tanh^2(u_i))}$$

Le terme $\log(1-\tanh^2(u))$ est **negatif** (car $1-\tanh^2 \in (0,1]$), donc le log-prob apres squashing est plus negatif que le log-prob gaussien. Sans cette correction, la loss acteur optimiserait une probabilite ne correspondant pas aux actions envoyees a l'environnement.

**Stabilite numerique.** Pour eviter $log(0)$ quand $|u|$ est grand, on utilise :
```python
log_prob = gauss_log_prob - (2*(math.log(2) - u - F.softplus(-2*u))).sum(-1)
```

In [ ]:
LOG_STD_MIN = -5
LOG_STD_MAX = 2


class SquashedGaussianActor(nn.Module):
    """Actor SAC : politique stochastique avec reparameterization + tanh squashing.

    Architecture :
      trunk      : Linear(obs_dim, hidden) -> ReLU -> Linear(hidden, hidden) -> ReLU
      mean_head  : Linear(hidden, action_dim)   # mu_theta(s)
      log_std_head: Linear(hidden, action_dim)  # log sigma_theta(s)

    Echantillonnage (reparameterization) :
      u = mu + sigma * eps,  eps ~ N(0, I)
      a = tanh(u)   (squashing pour borner dans [-1, 1])

    Log-prob avec correction tanh :
      log pi(a|s) = log pi_gauss(u|s) - sum_i log(1 - tanh^2(u_i))
    """

    def __init__(self, obs_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.fc1 = _orthogonal_init(nn.Linear(obs_dim, hidden_dim), gain=np.sqrt(2))
        self.fc2 = _orthogonal_init(nn.Linear(hidden_dim, hidden_dim), gain=np.sqrt(2))
        self.mean_head    = _orthogonal_init(nn.Linear(hidden_dim, action_dim), gain=0.01)
        self.log_std_head = _orthogonal_init(nn.Linear(hidden_dim, action_dim), gain=0.01)

    def forward(self, state):
        """Retourne (mean, log_std) de la gaussienne avant squashing."""
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mean    = self.mean_head(x)
        log_std = self.log_std_head(x).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def sample(self, state):
        """Echantillonne (action, log_prob, mean) via reparameterization + tanh.

        Returns
        -------
        action   : (batch, action_dim) in [-1, 1] apres tanh
        log_prob : (batch,) log pi(a|s) avec correction tanh
        mean     : (batch, action_dim) la moyenne (pour eval deterministe)
        """
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        u = normal.rsample()          # reparameterized sample
        action = torch.tanh(u)
        log_prob_gauss = normal.log_prob(u).sum(dim=-1)
        # Correction Jacobien tanh (formule numeriquement stable)
        log_prob = log_prob_gauss - (2 * (math.log(2) - u - F.softplus(-2 * u))).sum(dim=-1)
        return action, log_prob, mean

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        """Interface numpy pour le runner."""
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        mean, log_std = self.forward(obs_t)
        if deterministic:
            action = torch.tanh(mean)
        else:
            std = log_std.exp()
            u = torch.distributions.Normal(mean, std).rsample()
            action = torch.tanh(u)
        return action.squeeze(0).cpu().numpy().astype(np.float32)

In [ ]:
# Mini-test : SquashedGaussianActor — shapes, bornes, log_prob fini

torch.manual_seed(0)
_obs_dim2, _act_dim2 = 5, 3
actor_test = SquashedGaussianActor(_obs_dim2, _act_dim2, hidden_dim=32)
states_test = torch.randn(8, _obs_dim2)

actions_test, log_probs_test, means_test = actor_test.sample(states_test)

assert actions_test.shape == (8, _act_dim2)
assert log_probs_test.shape == (8,)
assert means_test.shape == (8, _act_dim2)
assert torch.isfinite(log_probs_test).all(), 'log_prob contient NaN ou Inf'
assert torch.all(actions_test <=  1.0 + 1e-6)
assert torch.all(actions_test >= -1.0 - 1e-6)

print(f'actions  shape : {actions_test.shape}  OK')
print(f'log_prob shape : {log_probs_test.shape}  OK')
print(f'actions  range : [{actions_test.min().item():.3f}, {actions_test.max().item():.3f}]  (doit etre dans [-1, 1])')
print(f'log_prob range : [{log_probs_test.min().item():.3f}, {log_probs_test.max().item():.3f}]  (toujours fini)')
print(f'mean    range  : [{means_test.min().item():.3f}, {means_test.max().item():.3f}]  (non squashe)')

In [ ]:
# Visualisation : distribution gaussienne vs squashed gaussian

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
configs = [
    ('mean=0, std=0.3', 0.0, 0.3),
    ('mean=0, std=1.0', 0.0, 1.0),
    ('mean=0.5, std=0.5', 0.5, 0.5),
    ('mean=0, std=2.0', 0.0, 2.0),
]

for ax, (title, mu, sigma) in zip(axes.flat, configs):
    np.random.seed(0)
    u_samples = np.random.normal(mu, sigma, 50000)
    a_samples = np.tanh(u_samples)
    u_range   = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
    gauss_pdf = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((u_range-mu)/sigma)**2)

    ax2 = ax.twinx()
    ax2.plot(u_range, gauss_pdf, 'b--', lw=1.5, alpha=0.6, label='Gaussienne (u)')
    ax2.set_ylabel('Densite gaussienne', color='b', fontsize=9)
    ax2.tick_params(axis='y', colors='b')

    ax.hist(a_samples, bins=80, density=True, color='tomato', alpha=0.7, label='tanh(u)')
    ax.axvline(-1, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.axvline( 1, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlim(-1.3, 1.3)
    ax.set_xlabel('Action $a = \\tanh(u)$')
    ax.set_ylabel('Densite squashed')
    ax.set_title(title, fontsize=11)

plt.suptitle('Distribution Squashed Gaussian pour differents (mean, std)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Log-prob gaussien brut vs log-prob corrige par tanh

u = np.linspace(-4.0, 4.0, 500)
mean_g, std_g = 0.0, 1.0
log_prob_gauss = -0.5 * ((u - mean_g) / std_g)**2 - np.log(std_g * np.sqrt(2 * np.pi))
a_squashed = np.tanh(u)
log_det_jac = np.log(1.0 - a_squashed**2 + 1e-8)
log_prob_corrected = log_prob_gauss - log_det_jac

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(u, log_prob_gauss, label='log_prob gaussien brut', lw=2)
axes[0].plot(u, log_prob_corrected, label='log_prob apres correction tanh', lw=2)
axes[0].axvline(0, color='black', lw=0.8, alpha=0.4)
axes[0].set_title("Dans l'espace pre-tanh u")
axes[0].set_xlabel('u'); axes[0].set_ylabel('log_prob'); axes[0].legend()

axes[1].plot(a_squashed, log_prob_gauss, label='si on oublie la correction', lw=2)
axes[1].plot(a_squashed, log_prob_corrected, label='densite coherente avec a=tanh(u)', lw=2)
axes[1].axvline(-1, color='k', ls='--', lw=0.8, alpha=0.5)
axes[1].axvline( 1, color='k', ls='--', lw=0.8, alpha=0.5)
axes[1].set_title("Vu depuis l'espace action a")
axes[1].set_xlabel('a = tanh(u)'); axes[1].set_ylabel('log_prob'); axes[1].legend()

plt.tight_layout()
plt.show()

### Interpretation du squashing

- **std faible (0.3)** : la masse reste concentree, peu d'accumulation aux bords. La distribution squashee ressemble a une gaussienne tronquee.
- **std fort (2.0)** : la masse s'accumule aux extremes $\pm 1$ — la distribution devient quasi-bimodale. Le squashing est tres deformant.
- **Sans correction** : le log-prob ignore que tanh compresse les queues. La loss acteur optimiserait une probabilite incorrecte dans l'espace $u$, pas dans l'espace action reel.
- **Avec correction** : le log-prob est toujours plus negatif que le log-prob gaussien, surtout pres des bords. L'entropie est correctement calculee dans l'espace des actions bornees.

**Transition.** L'actor et les critics sont construits. Il reste le replay buffer et l'assemblage complet.

## 7. Replay buffer off-policy

SAC est **off-policy** : les transitions stockees peuvent provenir de n'importe quelle version passee de la politique. La soft Bellman equation reste valide hors-politique car la valeur douce $V(s')$ reutilise la **politique courante** — pas celle qui a genere la transition. Pour les details, voir le notebook 07 (TD3).

Le buffer SAC est identique au buffer DDPG/TD3 : FIFO, actions continues float32, retourne des tenseurs PyTorch.

In [ ]:
class ContinuousReplayBuffer:
    """Buffer de replay FIFO pour actions continues.

    Stocke (s, a, r, s', done) ou a est un vecteur float32.
    Retourne des tenseurs PyTorch pour le calcul de gradient.
    Identique au buffer DDPG/TD3.
    """

    def __init__(self, capacity):
        self._buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self._buffer.append((
            np.asarray(state,      dtype=np.float32),
            np.asarray(action,     dtype=np.float32),
            float(reward),
            np.asarray(next_state, dtype=np.float32),
            bool(done),
        ))

    def sample(self, batch_size):
        indices = np.random.choice(len(self._buffer), size=batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(
            *(self._buffer[i] for i in indices)
        )
        return (
            torch.tensor(np.array(states),      dtype=torch.float32),
            torch.tensor(np.array(actions),     dtype=torch.float32),
            torch.tensor(np.array(rewards),     dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(np.array(dones),       dtype=torch.float32),
        )

    def __len__(self):
        return len(self._buffer)

## 3 differences cles entre SAC et TD3

On peut lire SAC comme TD3 avec trois modifications precises, qui s'enchainent
logiquement :

| # | Aspect | TD3 | SAC | Pourquoi |
|---|--------|-----|-----|----------|
| 1 | **Action dans la cible** | $\mu_{\text{targ}}(s') + \epsilon$ clippe | $a' \sim \pi_\theta(\cdot\|s')$ | la stochasticite joue un role proche du smoothing |
| 2 | **Cible Bellman** | $\min Q_{\text{targ}}(s', a')$ | $\min Q_{\text{targ}} - \alpha \log \pi$ | l'entropie entre dans la valeur cible |
| 3 | **Loss acteur** | $-\mathbb{E}[Q1(s, \mu_\theta(s))]$ | $\mathbb{E}[\alpha \log \pi - \min Q]$ | l'acteur cherche aussi une politique incertaine utile |

**Nuance importante.** Dire que SAC n'a plus besoin de `policy_delay` ou de
`target actor` est une propriete de cette recette particuliere, pas une loi
universelle sur toutes les variantes actor-critic continues. Ici, la politique
stochastique et le terme d'entropie changent suffisamment la geometrie de
l'optimisation pour que la version canonique SAC se passe de ces deux ingredients.

**Transition.** On a maintenant tous les ingredients. Il est temps d'assembler l'agent.

In [ ]:
class SACAgent:
    """Agent SAC : actor stochastique + twin critics + auto-alpha.

    Trois optimiseurs : actor, critic, alpha.
    Pas d'actor_target (la stochasticite joue le role de target smoothing).
    """

    def __init__(self, obs_dim, action_dim, *, hidden_dim=256,
                 actor_lr=3e-4, critic_lr=3e-4, alpha_lr=3e-4,
                 gamma=0.99, tau=0.005,
                 buffer_capacity=1_000_000, batch_size=256,
                 init_alpha=0.2, auto_tune_alpha=True,
                 target_entropy=None):
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.auto_tune_alpha = auto_tune_alpha

        # Reseaux
        self.actor  = SquashedGaussianActor(obs_dim, action_dim, hidden_dim)
        self.critic = TwinQNetwork(obs_dim, action_dim, hidden_dim)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_target.eval()

        # Optimiseurs
        self.actor_optimizer  = optim.Adam(self.actor.parameters(),  lr=actor_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)

        # Temperature alpha — apprise via log_alpha pour garantir alpha > 0
        self.log_alpha = torch.tensor(math.log(init_alpha), requires_grad=True, dtype=torch.float32)
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=alpha_lr)
        self.target_entropy = target_entropy if target_entropy is not None else -float(action_dim)

        # Replay buffer
        self.replay_buffer = ContinuousReplayBuffer(buffer_capacity)

    @property
    def alpha(self):
        """Temperature courante (toujours > 0 car exp)."""
        return self.log_alpha.exp().item()

    def select_action(self, obs, deterministic=False):
        """Choisit une action depuis la politique stochastique (ou deterministe pour eval)."""
        return self.actor.act(obs, deterministic=deterministic)

    def store_transition(self, state, action, reward, next_state, done):
        """Enregistre (s, a, r, s', done) dans le buffer."""
        self.replay_buffer.push(state, action, reward, next_state, done)

    def _compute_critic_loss(self, states, actions, rewards, next_states, dones, alpha):
        """Soft Bellman error pour les deux critiques.

        Cible : y = r + gamma*(1-d)*[min(Q1_targ, Q2_targ)(s', a~') - alpha*log_pi(a~'|s')]
        a~' ~ pi_theta(.|s')  (politique courante, PAS une cible)
        """
        with torch.no_grad():
            next_actions, next_log_probs, _ = self.actor.sample(next_states)
            q1_next, q2_next = self.critic_target(next_states, next_actions)
            q_next  = torch.min(q1_next, q2_next) - alpha.detach() * next_log_probs
            q_target = rewards + self.gamma * (1.0 - dones) * q_next

        q1, q2 = self.critic(states, actions)
        return F.mse_loss(q1, q_target) + F.mse_loss(q2, q_target)

    def _compute_actor_loss(self, states, alpha):
        """Loss acteur SAC : -E[min Q(s, a~) - alpha * log_pi(a~|s)].

        Reparameterization : a~ = mu + sigma * eps, grad passe a travers a~.
        """
        actions_pi, log_probs, _ = self.actor.sample(states)
        q1_pi, q2_pi = self.critic(states, actions_pi)
        min_q_pi = torch.min(q1_pi, q2_pi)
        actor_loss = (alpha.detach() * log_probs - min_q_pi).mean()
        return actor_loss, log_probs.mean().item()

    def _compute_alpha_loss(self, states):
        """Loss pour l'auto-tuning de alpha.

        Objectif dual : min_alpha -alpha * E[log_pi + target_entropy]
        Si entropie > target : alpha diminue. Si entropie < target : alpha augmente.
        """
        with torch.no_grad():
            _, log_probs, _ = self.actor.sample(states)
        return -(self.log_alpha * (log_probs + self.target_entropy)).mean()
    
    def learn_step(self):
        """Mise a jour SAC : critic -> actor -> alpha -> soft update."""
        if len(self.replay_buffer) < self.batch_size:
            return {}

        states, actions, rewards, next_states, dones = \
            self.replay_buffer.sample(self.batch_size)

        alpha = self.log_alpha.exp()  # tenseur pour le gradient alpha

        # 1. Mise a jour des deux critiques
        critic_loss = self._compute_critic_loss(states, actions, rewards, next_states, dones, alpha)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # 2. Mise a jour de l'acteur
        actor_loss, log_prob_mean = self._compute_actor_loss(states, alpha)
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # 3. Mise a jour de alpha (auto-tuning optionnel)
        if self.auto_tune_alpha:
            alpha_loss = self._compute_alpha_loss(states)
            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()
        else:
            alpha_loss = torch.tensor(0.0)

        # 4. Soft update des critiques cibles (Polyak averaging)
        self._soft_update(self.critic, self.critic_target, self.tau)

        with torch.no_grad():
            q1, q2 = self.critic(states, actions)

        return {
            'critic_loss': critic_loss.item(),
            'actor_loss':  actor_loss.item(),
            'alpha_loss':  alpha_loss.item(),
            'alpha':       self.alpha,
            'entropy':    -log_prob_mean,
            'log_prob_mean': log_prob_mean,
            'q1_mean':    q1.mean().item(),
            'q2_mean':    q2.mean().item(),
            'q_gap':      (q1 - q2).abs().mean().item(),
        }

    @staticmethod
    def _soft_update(online, target, tau):
        """Polyak averaging : target <- tau*online + (1-tau)*target."""
        for p_on, p_targ in zip(online.parameters(), target.parameters()):
            p_targ.data.copy_(tau * p_on.data + (1.0 - tau) * p_targ.data)

    def save(self, path):
        torch.save({'actor': self.actor.state_dict(),
                    'critic': self.critic.state_dict(),
                    'log_alpha': self.log_alpha.item()}, path)

    def load(self, path):
        ckpt = torch.load(path, map_location='cpu', weights_only=True)
        self.actor.load_state_dict(ckpt['actor'])
        self.critic.load_state_dict(ckpt['critic'])
        with torch.no_grad():
            self.log_alpha.fill_(ckpt['log_alpha'])

### Les 3 losses SAC

Chaque mise a jour `learn_step` calcule trois losses dans l'ordre :

| # | Loss | Objectif |
|---|------|----------|
| 1 | **Critic** | Minimiser l'erreur de Bellman soft : $\mathbb{E}[(Q_i(s,a) - y)^2]$ |
| 2 | **Actor** | Maximiser $\mathbb{E}[\min Q(s,\tilde{a}) - \alpha\log\pi(\tilde{a}\|s)]$ via reparameterization |
| 3 | **Alpha** | Maintenir $\mathbb{E}[-\log\pi] \geq \bar{\mathcal{H}}$ via dualite de Lagrange |

In [ ]:
# Mini-test de l'agent SAC sur donnees synthetiques

_test_obs, _test_act = 5, 3
_test_batch = 8
rng_test = np.random.default_rng(0)

agent_test = SACAgent(_test_obs, _test_act, hidden_dim=32,
                      buffer_capacity=100, batch_size=_test_batch, init_alpha=0.2)

for _ in range(20):
    s  = rng_test.standard_normal(_test_obs).astype(np.float32)
    a  = rng_test.uniform(-1, 1, size=_test_act).astype(np.float32)
    r  = float(rng_test.standard_normal())
    sn = rng_test.standard_normal(_test_obs).astype(np.float32)
    agent_test.store_transition(s, a, r, sn, False)

obs_dummy = np.zeros(_test_obs, dtype=np.float32)
act = agent_test.select_action(obs_dummy, deterministic=False)
assert act.shape == (_test_act,), f'action shape: {act.shape}'
assert np.all(np.abs(act) <= 1.0 + 1e-6), 'action hors bornes'

alpha_before = agent_test.alpha
metrics = agent_test.learn_step()
required = {'critic_loss','actor_loss','alpha_loss','alpha','entropy','q1_mean','q2_mean','q_gap'}
assert required.issubset(metrics.keys())
assert all(np.isfinite(metrics[k]) for k in required), 'une metrique est NaN'

print(f'select_action shape   : {act.shape}  OK')
print(f'select_action range   : [{act.min():.3f}, {act.max():.3f}]  OK')
print(f'learn_step metriques  : {sorted(metrics.keys())}')
print(f'alpha avant/apres     : {alpha_before:.4f} -> {metrics["alpha"]:.4f}')
print(f'critic_loss           : {metrics["critic_loss"]:.4f}')
print(f'entropy               : {metrics["entropy"]:.4f}')
print('Mini-test SAC OK')

## 8. Pseudocode SAC (SpinningUp)

---
$$
\boxed{\begin{aligned}
&\textbf{Algorithme : Soft Actor-Critic (SAC)}\\
&\textbf{Entree :} \text{parametres initiaux } \theta,\ \phi_1,\ \phi_2,\ \text{buffer vide } \mathcal{D},\ \alpha\\
&\textbf{Init :} \phi_{targ,1} \leftarrow \phi_1,\ \phi_{targ,2} \leftarrow \phi_2\ \text{(pas de target actor)}\\
&\textbf{Repeter :}\\
&\quad 1.\ a \sim \pi_\theta(\cdot|s)\ \text{(echantillonnage stochastique)}\\
&\quad 2.\ \text{executer } a,\ \text{observer } s', r, d\\
&\quad 3.\ \text{stocker } (s, a, r, s', d) \text{ dans } \mathcal{D}\\
&\quad \textbf{Si c'est le moment de mettre a jour :}\\
&\quad\quad a.\ \text{echantillonner } B \text{ depuis } \mathcal{D}\\
&\quad\quad b.\ \tilde{a}' \sim \pi_\theta(\cdot|s')\ \text{(politique courante, pas cible)}\\
&\quad\quad c.\ y = r + \gamma(1-d)[\min_i Q_{\phi_{targ,i}}(s', \tilde{a}') - \alpha\log\pi_\theta(\tilde{a}'|s')]\\
&\quad\quad d.\ \nabla_{\phi_i} \frac{1}{|B|}\sum (Q_{\phi_i}(s,a) - y)^2,\ i=1,2\\
&\quad\quad e.\ \nabla_\theta \frac{1}{|B|}\sum [\min_i Q_{\phi_i}(s, \tilde{a}_\theta) - \alpha\log\pi_\theta(\tilde{a}_\theta|s)]\\
&\quad\quad f.\ \nabla_\alpha \frac{1}{|B|}\sum[-\alpha(\log\pi_\theta(\tilde{a}|s) + \bar{\mathcal{H}})]\\
&\quad\quad g.\ \phi_{targ,i} \leftarrow \rho\,\phi_{targ,i} + (1-\rho)\,\phi_i\ \text{(Polyak)}
\end{aligned}}
$$

> **Note :** SpinningUp utilise $\rho \approx 1$ pour le Polyak averaging ; dans le code on utilise $\tau = 1 - \rho = 0.005$.

## 9. L'agent SAC : description de l'architecture

L'agent SAC assemble les composants suivants :

- **`actor`** : `SquashedGaussianActor` — la politique stochastique
- **`critic`** : `TwinQNetwork` — deux Q-networks independants
- **`critic_target`** : copie figee du critic, mise a jour en douceur (Polyak)
- **`log_alpha`** : parametre scalaire appris, `alpha = exp(log_alpha)`

Trois optimiseurs separes :
1. `actor_optimizer` — pour $\theta$
2. `critic_optimizer` — pour $\phi_1, \phi_2$
3. `alpha_optimizer` — pour $\log\alpha$

La methode `learn_step` coordonne les quatre mises a jour dans l'ordre :
1. Critic loss (soft Bellman error)
2. Actor loss (reparameterization + entropie)
3. Alpha loss (auto-tuning)
4. Soft update des critic targets (Polyak)

### Flux SAC : replay, twin critics, acteur et temperature

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    classDef result fill:none,stroke:#16a34a,stroke-width:2px

    env["Environnement"]:::primary
    replay["Replay buffer"]:::primary
    critics["Twin critics<br/>$$Q_{\phi_1}, Q_{\phi_2}$$"]:::secondary
    actor["Acteur stochastique<br/>$$\pi_\theta$$"]:::primary
    env -->|"transition réelle"| replay
    replay -->|"batch de transitions"| critics
    replay -->|"états courant et suivant"| actor
    actor -->|"$$\tilde{a}, \log \pi$$"| critics
    critics --> critic_step["Soft Bellman update"]:::result
    actor --> actor_step["Policy update $$\alpha \log \pi - \min Q$$"]:::result
    actor --> alpha_step["Temperature update $$\log \alpha$$"]:::result
    alpha_step --> actor
    critic_step --> targets["Polyak sur les cibles critiques"]:::result
```

## 10. Hyperparametres SAC

| Parametre | Valeur | Justification |
|-----------|--------|---------------|
| `hidden_dim` | 256 | Standard MuJoCo (2 couches cachees) |
| `actor_lr` | 3e-4 | Learning rate acteur (Adam) |
| `critic_lr` | 3e-4 | Learning rate critique (Adam) |
| `alpha_lr` | 3e-4 | Learning rate temperature (Adam) |
| `gamma` | 0.99 | Facteur d'actualisation |
| `tau` | 0.005 | Coefficient de Polyak averaging |
| `buffer_capacity` | 100 000 | Taille du replay buffer (reduit pour le notebook) |
| `batch_size` | 256 | Taille des mini-batches |
| `init_alpha` | 0.2 | Temperature initiale (sera ajustee automatiquement) |
| `target_entropy` | -6 | = -action_dim, heuristique standard SpinningUp |
| `start_steps` | 5 000 | Pas d'exploration aleatoire (warm-up) |
| `update_after` | 1 000 | Attendre ce nombre de transitions avant de commencer les mises a jour |

> **Note `terminated` vs `truncated`** : on stocke `done = terminated` (pas `truncated`) dans le buffer. Pour `HalfCheetah-v5`, les episodes finissent surtout par limite de temps (`truncated=True`, `terminated=False`). Stocker `done=1` pour truncation couperait artificiellement la cible de Bellman en fin d'episode.

In [ ]:
# Configuration de l'expérience SAC sur HalfCheetah.
SEED = 42
ENV_ID = "HalfCheetah-v5"
EVAL_SEED = SEED

TOTAL_STEPS = 100_000
START_STEPS = 5_000
UPDATE_AFTER = 1_000
EVAL_EVERY = 10_000
EVAL_EPISODES = 5
MAX_STEPS_PER_EPISODE = 1000

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
def evaluate(agent, env_id, num_episodes=5, seed=0):
    """Evalue l'agent en mode deterministe sur plusieurs episodes.

    Pour SAC, le mode deterministe = tanh(mean) sans echantillonnage.
    """
    env = gym.make(env_id)
    rewards = []
    try:
        for i in range(num_episodes):
            obs, _ = env.reset(seed=seed + i)
            total_reward = 0.0
            for _ in range(MAX_STEPS_PER_EPISODE):
                action = agent.select_action(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                if terminated or truncated:
                    break
            rewards.append(total_reward)
    finally:
        env.close()
    return np.mean(rewards), np.std(rewards)


def train_sac():
    """La boucle garde l'orchestration explicite : env -> replay -> learn_step -> eval."""
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    env = gym.make(ENV_ID)
    env.action_space.seed(SEED)
    agent = SACAgent(
        OBS_DIM, ACTION_DIM,
        hidden_dim=256,
        actor_lr=3e-4, critic_lr=3e-4, alpha_lr=3e-4,
        gamma=0.99, tau=0.005,
        buffer_capacity=100_000, batch_size=256,
        init_alpha=0.2, auto_tune_alpha=True,
        target_entropy=-ACTION_DIM,
    )

    sac_history = {
        'episode_rewards': [],
        'eval_steps': [], 'eval_mean_rewards': [], 'eval_std_rewards': [],
        'step_critic_losses': [], 'step_actor_losses': [], 'step_alpha_losses': [],
        'step_alphas': [], 'step_entropies': [], 'step_log_prob_means': [],
        'step_q1_means': [], 'step_q2_means': [], 'step_q_gaps': [],
    }

    total_steps = 0
    episode = 0
    try:
        while total_steps < TOTAL_STEPS:
            obs, _ = env.reset(seed=SEED + episode)
            episode_reward = 0.0
            for _ in range(MAX_STEPS_PER_EPISODE):
                if total_steps < START_STEPS:
                    action = env.action_space.sample()
                else:
                    action = agent.select_action(obs, deterministic=False)

                next_obs, reward, terminated, truncated, _ = env.step(action)
                episode_done = terminated or truncated
                agent.store_transition(obs, action, reward, next_obs, float(terminated))
                episode_reward += reward
                obs = next_obs
                total_steps += 1

                if total_steps >= UPDATE_AFTER:
                    metrics = agent.learn_step()
                    if metrics:
                        sac_history['step_critic_losses'].append(metrics['critic_loss'])
                        sac_history['step_actor_losses'].append(metrics['actor_loss'])
                        sac_history['step_alpha_losses'].append(metrics['alpha_loss'])
                        sac_history['step_alphas'].append(metrics['alpha'])
                        sac_history['step_entropies'].append(metrics['entropy'])
                        sac_history['step_log_prob_means'].append(metrics['log_prob_mean'])
                        sac_history['step_q1_means'].append(metrics['q1_mean'])
                        sac_history['step_q2_means'].append(metrics['q2_mean'])
                        sac_history['step_q_gaps'].append(metrics['q_gap'])

                if total_steps % EVAL_EVERY == 0:
                    eval_mean, eval_std = evaluate(agent, ENV_ID, EVAL_EPISODES, seed=EVAL_SEED)
                    sac_history['eval_steps'].append(total_steps)
                    sac_history['eval_mean_rewards'].append(eval_mean)
                    sac_history['eval_std_rewards'].append(eval_std)
                    print(f"[SAC] Step {total_steps:>7d} | Eval: {eval_mean:.1f} +/- {eval_std:.1f} | alpha: {agent.alpha:.4f}")

                if episode_done:
                    break

            sac_history['episode_rewards'].append(episode_reward)
            episode += 1
            if episode % 20 == 0:
                avg = np.mean(sac_history['episode_rewards'][-20:])
                print(f"[SAC] Episode {episode:>4d} | Steps {total_steps:>7d} | Avg20: {avg:.1f} | alpha: {agent.alpha:.4f}")
    finally:
        env.close()

    print(f"\nSAC termine : {episode} episodes, {total_steps} pas")
    return agent, sac_history

In [ ]:
agent, sac_history = train_sac()


In [ ]:
# Chargement des resultats existants, sans relancer le training

def find_existing_sac_histories():
    roots = [Path.cwd(), Path.cwd().parent]
    histories = []
    for root in roots:
        runs_dir = root / 'runs' / 'sac'
        if runs_dir.exists():
            histories.extend(sorted(runs_dir.glob('*/history.json')))
    unique = {p.resolve(): p for p in histories}
    return sorted(unique.values(), key=lambda path: path.stat().st_mtime, reverse=True)

if sac_history is None:
    existing = find_existing_sac_histories()
    if existing:
        with existing[0].open('r') as f:
            sac_history = json.load(f)
        print(f"Run SAC charge : {existing[0]}")
        print("Lecture prudente : ces courbes documentent une run locale existante, pas une re-execution exacte de ce notebook.")
    else:
        print("Aucune run SAC trouvee. Lancer RUN_SAC_TRAINING = True.")

## Transition : lire les courbes

SAC expose des metriques que DDPG/TD3 n'ont pas : `alpha`, l'entropie, et la loss d'alpha. Ces metriques aident a comprendre **pourquoi** l'agent apprend ou bloque, pas seulement s'il apprend.

In [ ]:
def moving_average(data, window=10):
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window) / window, mode='valid')


if sac_history:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    color_sac = '#2ca02c'
    window = 15

    # (1) Recompenses d'entrainement
    ax = axes[0, 0]
    ep_r = sac_history['episode_rewards']
    ma = moving_average(ep_r, window)
    ax.plot(ep_r, alpha=0.2, color=color_sac, lw=0.8)
    ax.plot(range(window - 1, len(ep_r)), ma, color=color_sac, lw=2,
            label=f'Moy. glissante ({window} ep.)')
    ax.set_title('Recompense entrainement')
    ax.set_xlabel('Episode'); ax.set_ylabel('Recompense totale'); ax.legend(fontsize=9)

    # (2) Evaluation deterministe
    ax = axes[0, 1]
    if sac_history['eval_steps']:
        ax.errorbar(sac_history['eval_steps'], sac_history['eval_mean_rewards'],
                    yerr=sac_history['eval_std_rewards'],
                    color=color_sac, marker='o', capsize=4, label='Eval SAC')
    ax.set_title('Evaluation deterministe')
    ax.set_xlabel('Pas'); ax.set_ylabel('Reward moyen (5 ep.)'); ax.legend(fontsize=9)

    # (3) Temperature alpha
    ax = axes[0, 2]
    ax.plot(sac_history['step_alphas'], color='darkorange', lw=1.5, label='alpha')
    ax.set_title('Temperature alpha (auto-tune)')
    ax.set_xlabel('Update'); ax.set_ylabel('alpha'); ax.legend(fontsize=9)

    # (4) Entropie vs cible
    ax = axes[1, 0]
    ax.plot(sac_history['step_entropies'], color='royalblue', lw=1.5, label='Entropie courante')
    ax.axhline(ACTION_DIM, color='k', ls='--', lw=1.5, label=f'Cible ~{ACTION_DIM}')
    ax.set_title('Entropie H(pi) vs cible')
    ax.set_xlabel('Update'); ax.set_ylabel('Entropie'); ax.legend(fontsize=9)

    # (5) Critic loss
    ax = axes[1, 1]
    ax.plot(sac_history['step_critic_losses'], color='tomato', lw=1.0, alpha=0.8,
            label='Critic loss')
    ma_c = moving_average(sac_history['step_critic_losses'], 50)
    ax.plot(range(49, len(sac_history['step_critic_losses'])), ma_c,
            color='darkred', lw=2, label='Moy. 50')
    ax.set_title('Perte critique (MSE Bellman)')
    ax.set_xlabel('Update'); ax.set_yscale('symlog'); ax.legend(fontsize=9)

    # (6) Actor loss
    ax = axes[1, 2]
    ax.plot(sac_history['step_actor_losses'], color='purple', lw=1.0, alpha=0.8,
            label='Actor loss')
    ma_a = moving_average(sac_history['step_actor_losses'], 50)
    ax.plot(range(49, len(sac_history['step_actor_losses'])), ma_a,
            color='indigo', lw=2, label='Moy. 50')
    ax.set_title('Perte acteur (alpha*log_pi - Q)')
    ax.set_xlabel('Update'); ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print('sac_history est None — lancer le training pour visualiser.')

## Lecture des courbes SAC — panneau par panneau

### Recompenses d'entrainement (haut gauche)
Collectees avec une politique stochastique, donc plus bruitees que l'eval deterministe. La moyenne glissante montre la tendance generale. Sur 100 000 pas, on observe surtout des tendances ; un vrai jugement demande plus de timesteps et plusieurs seeds.

### Evaluation deterministe (haut centre)
C'est la metrique de reference. Elle utilise `tanh(mean)` sans echantillonnage. Une progression monotone est rassurante ; des plateaux indiquent un point de blocage a investiguer avec les autres metriques.

### Temperature alpha (haut droite)
alpha peut baisser si la politique est trop entropique, ou monter si l'entropie est sous la cible. Une courbe stable est rassurante, mais une courbe qui bouge n'est pas automatiquement mauvaise : l'auto-tuning reagit aux changements de la politique.

### Entropie vs cible (bas gauche)
Si on trace `entropy = -log_prob_mean`, la cible visuelle est environ `+action_dim` (ici ~6). Si l'entropie diverge tres loin de la cible pendant longtemps, `alpha_lr` ou `target_entropy` sont probablement a revoir.

### Critic loss (bas centre)
Bruitee parce que les cibles Bellman bougent avec la politique courante. On cherche surtout a eviter les explosions longues et les NaN. Une echelle symlog aide a voir les fluctuations.

### Actor loss (bas droite)
La loss acteur SAC est $\alpha\log\pi - \min Q$. Elle ne se lit pas comme une loss supervisee classique : elle peut etre positive ou negative. Elle sert surtout a detecter des anomalies, pas a declarer seule que la politique est bonne.

In [ ]:
# Diagnostics Q-values : Q1, Q2, gap

if sac_history and sac_history.get('step_q1_means'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    color_q1, color_q2 = '#1f77b4', '#ff7f0e'

    ax = axes[0]
    ax.plot(sac_history['step_q1_means'], color=color_q1, lw=1.2, alpha=0.7, label='Q1 mean')
    ma_q1 = moving_average(sac_history['step_q1_means'], 50)
    ax.plot(range(49, len(sac_history['step_q1_means'])), ma_q1, color=color_q1, lw=2.5,
            label='Moy. 50')
    ax.set_title('Q1 moyen sur les batches')
    ax.set_xlabel('Update'); ax.set_ylabel('Q1 moyen'); ax.legend(fontsize=9)

    ax = axes[1]
    ax.plot(sac_history['step_q2_means'], color=color_q2, lw=1.2, alpha=0.7, label='Q2 mean')
    ma_q2 = moving_average(sac_history['step_q2_means'], 50)
    ax.plot(range(49, len(sac_history['step_q2_means'])), ma_q2, color=color_q2, lw=2.5,
            label='Moy. 50')
    ax.set_title('Q2 moyen sur les batches')
    ax.set_xlabel('Update'); ax.set_ylabel('Q2 moyen'); ax.legend(fontsize=9)

    ax = axes[2]
    ax.plot(sac_history['step_q_gaps'], color='gray', lw=1.0, alpha=0.6, label='|Q1-Q2|')
    ma_gap = moving_average(sac_history['step_q_gaps'], 50)
    ax.plot(range(49, len(sac_history['step_q_gaps'])), ma_gap, color='black', lw=2,
            label='Moy. 50')
    ax.set_title('Gap |Q1-Q2| (desaccord des critiques)')
    ax.set_xlabel('Update'); ax.set_ylabel('|Q1-Q2|'); ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print('Pas de donnees Q disponibles — lancer le training pour voir ces courbes.')

### Interpretation des diagnostics Q

- **Q1 et Q2 montent** : normal si l'agent apprend a mieux evaluer les etats. Q monte avec la politique.
- **Q monte tres vite sans amelioration de l'eval** : signe d'overestimation — l'actor exploite une erreur du critic.
- **Petit gap |Q1-Q2|** : les deux critiques sont d'accord. Bonne signe de stabilite.
- **Grand gap |Q1-Q2|** : les critiques divergent. A surveiller si le gap augmente indefiniment — peut indiquer un manque de diversite dans le buffer ou un learning rate trop eleve.

En pratique : regarder Q, alpha, entropie et eval **ensemble**, pas isolement.

In [ ]:
# === Démonstration finale : replay vidéo de la politique SAC déterministe ===
def evaluate_and_display_agent(agent, label="SAC", n_episodes=3, seed=EVAL_SEED, max_steps=MAX_STEPS_PER_EPISODE, video_root="videos/08_sac_halfcheetah"):
    """Évalue SAC en mode déterministe et affiche le dernier replay vidéo enregistré."""
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    rewards = []
    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=seed + ep)
            total = 0.0
            steps = 0
            for _ in range(max_steps):
                action = agent.select_action(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                total += float(reward)
                steps += 1
                if terminated or truncated:
                    break
            rewards.append(total)
            print(f"[{label}] Épisode {ep + 1} : retour={total:+.1f} ({steps} pas)")
    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):+.1f} +/- {np.std(rewards):.1f}")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos and Video is not None and display is not None:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=640))
    elif not videos:
        print(f"Aucune vidéo générée pour {label}.")
    else:
        print(f"Vidéo générée mais affichage IPython indisponible : {videos[-1]}")

    return rewards


if agent and sac_history:
    sac_demo_rewards = evaluate_and_display_agent(agent, "SAC")
else:
    print("SAC non entraîné -- pas de démonstration vidéo.")


## 12. Comparaison SAC vs TD3 vs DDPG

| Dimension | DDPG | TD3 | SAC |
|-----------|------|-----|-----|
| **Politique** | Deterministe $\mu_\theta(s)$ | Deterministe $\mu_\theta(s)$ | Stochastique $\pi_\theta(\cdot\|s)$ |
| **Exploration** | Bruit gaussien externe | Bruit gaussien externe | Entropie intrinseque |
| **Critics** | Single Q | Twin Q | Twin soft Q |
| **Target actor** | oui | oui | non dans la version canonique |
| **Target smoothing** | non | oui | remplace par l'echantillonnage stochastique |
| **Policy delay** | non | oui | non dans la version canonique |
| **Temperature** | non | non | $\alpha$ appris |
| **Objectif** | max $\mathbb{E}[r]$ | max $\mathbb{E}[r]$ | max $\mathbb{E}[r + \alpha H(\pi)]$ |
| **Overestimation** | forte | attenuee | attenuee |
| **Stabilite** | moderee | bonne | souvent robuste, mais sensible a $\alpha$ et au replay |
| **Sample efficiency** | moderee | bonne | souvent competitive, sans garantie universelle |
| **Nb optimiseurs** | 2 | 2 | 3 |
| **Nb reseaux** | 4 | 5 | 5 (`actor`, twin critics, twin targets, `log_alpha`) |

## 13. Recapitulatif du parcours off-policy continu

### La progression des algorithmes

$$\underbrace{\text{DQN}}_{\substack{\text{discret} \\ \text{off-policy}}} \to \underbrace{\text{DDPG}}_{\substack{\text{continu deterministe} \\ \text{+ bruit externe}}} \to \underbrace{\text{TD3}}_{\substack{\text{+ twin critics} \\ \text{+ policy delay} \\ \text{+ target smoothing}}} \to \underbrace{\text{SAC}}_{\substack{\text{stochastique} \\ \text{+ entropie intrinseque} \\ \text{+ alpha auto-tune}}}$$

Chaque etape resout un probleme de la precedente :

| Etape | Probleme resolu | Mecanisme |
|-------|----------------|----------|
| DDPG | DQN ne fonctionne pas en continu | acteur deterministe + gradient DPG |
| TD3 | DDPG est instable (overestimation, bruit gradient) | twin Q + delay + smoothing |
| SAC | TD3 explore moins richement et fixe implicitement le compromis exploration/exploitation | politique stochastique + entropie + auto-alpha |

SAC est le seul algorithme de cette progression a changer explicitement l'objectif
optimise. Ce n'est donc pas seulement un TD3 plus stable : c'est une autre maniere de
definir ce qu'une bonne politique doit faire pendant l'apprentissage.

## References

- Haarnoja, T., Zhou, A., Abbeel, P. & Levine, S. (2018). Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor. *ICML 2018*. [arXiv:1801.01290](https://arxiv.org/abs/1801.01290).

- Haarnoja, T., Zhou, A., Hartikainen, K. et al. (2018). Soft Actor-Critic Algorithms and Applications. *arXiv:1812.05905*. [arXiv:1812.05905](https://arxiv.org/abs/1812.05905). (SAC v2 : auto-tuning de alpha, suppression du V-network separe)

- Fujimoto, S., van Hoof, H. & Meger, D. (2018). Addressing Function Approximation Error in Actor-Critic Methods. *ICML 2018*. [arXiv:1802.09477](https://arxiv.org/abs/1802.09477).

- Lillicrap, T. P. et al. (2015). Continuous control with deep reinforcement learning. *ICLR 2016*. [arXiv:1509.02971](https://arxiv.org/abs/1509.02971).

- OpenAI Spinning Up. *Soft Actor-Critic*. [spinningup.openai.com](https://spinningup.openai.com/en/latest/algorithms/sac.html).